# Step 23.3 — Training module

**命令一：**

~~~bash
git pull
~~~

**命令二：**

~~~bash
cp -n notebooks/104_step23_3_training_module.ipynb notebooks/work/104_step23_3_training_module_$(date +%Y%m%d_%H%M%S)_work.ipynb
~~~

只在独立 work 副本运行/修改；本轮只静态准备，未执行任何 Notebook、MLX、训练、生成或 benchmark。Apple Silicon / Python 3.12 / MLX v0.32.2。


## 1. 动机（Motivation）

前面 17.5、19.2、19.3 已把 loss、gradient、Adam、validation 放在 Notebook。本节将这些接口迁到 src/minigpt/train.py；新增的是函数之间的清晰契约，不是更大的模型或已执行训练。仍使用 uncompiled step，便于看清显式 mx.eval 边界与变化的 optimizer state。

完整运行实现放在 **src/minigpt/train.py**，以下 Notebook 只教学本小节新增接口，不复制旧类的长代码。


## 2. Shape / Input–Output Contract

输入：X/Y 都是 (B,T) int32；MiniGPT 产生 logits (B,T,V)。将 logits 展平为 (B*T,V)，targets 展平为 (B*T,)，cross_entropy 后均值为标量 ()。nn.value_and_grad 返回 model 参数树的梯度；global_grad_norm 把所有梯度叶子平方归约为一个标量。make_train_step 绑定同一 model 和 Adam，返回 (loss_before_update, preclip_norm)，各 shape ()。验证函数只前向，不更新 Adam。


## 3. 手算小例子（Paper example, not runtime output）

令 B=1,T=2,V=3，原 logits 为两个全 0 行，targets=[0,2]。每位置 softmax 为 [1/3,1/3,1/3]，交叉熵平均为 ln(3)。这是纸上数值，不代表随机模型执行输出。该 loss 是当前参数更新**之前**的版本；下一次 validation 用的是更新后的参数。


In [ ]:
# Only future independent work copy; this code is not executed now.
# 从 root/notebooks/notebooks/work 向上定位源包。
from pathlib import Path
import sys

repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file()
     and (p / "src" / "minigpt" / "__init__.py").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("MiniGPT repository not found above kernel working directory")
src_dir = str(repo_root / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
from minigpt.train import language_model_loss, make_train_step, run_training
# Core contract:
# train_step = make_train_step(model, optimizer, max_grad_norm=1.0)
# loss_before_update, norm = train_step(X, Y)
# records = run_training(model, train_step, train_tokens, ...)
# No model, optimizer, data, or training call happens in this master.


## 4. 限制、预期验证与静态结论

Validation batches require an independently owned random.Random; training sampling must not reuse validation RNG. max_grad_norm=None disables clipping; a positive finite value enables global clipping. Nonfinite loss/grad norm abort before optimizer.update, but other runtime failures are not promised away. The returned metrics are Python numbers only when explicitly called. No torch, autocast, graph compilation, automatic checkpoint, no fabricated loss curve.

未来可验证的前提是合适的真实模型、正确的数组和显式调用；静态审核无法保证任意设备上通过，也没有任何读数、模型生成或训练记录。

**练习（不附答案）：** ① 列出 B=4,T=8,V=32 时展平前后 shapes；② 解释为什么返回 old-parameter loss 不等于更新后的 validation loss；③ 画出训练 RNG 和验证 RNG 的分开调用路线。

本节材料初稿，实际修正 0/2；按用户新授权自动继续下一小节。


**工作副本的导入位置（Working-directory discovery）**：下列代码在仓库 root、`notebooks/` 或 `notebooks/work/` 启动 kernel 时，均从 `Path.cwd()` 向上查找同时含 `pyproject.toml` 与 `src/minigpt/__init__.py` 的目录，再将其 `src` 加入 `sys.path`。这不是依赖单一 cwd 的硬编码路径；如果 kernel 在仓库树外启动，将明确抛出 `FileNotFoundError`。这段逻辑只负责定位代码，**没有运行课程代码，也不是自动安装**。

### 23.3 真实 API / 时间线（教学调用，不执行）
假设调用前已经分别准备好同一组 `model`、`optimizer`、`train_tokens`、`val_tokens`；以下为与 `src/minigpt/train.py` 函数签名对齐的**未来调用片段**，不是本 Notebook 自带模型或训练入口：

```python
from random import Random
train_rng, val_rng = Random(2026), Random(2027)  # 两套独立 Python RNG 状态
train_step = make_train_step(model, optimizer, max_grad_norm=1.0)
history = run_training(
    model, train_step, train_tokens,
    batch_size=4, block_size=64, updates=4, train_rng=train_rng,
    val_tokens=val_tokens, val_rng=val_rng,
    val_every=2, val_batches=2,
)
```

| 循环内 index | `record["update"]` | 假设进入循环前 Adam step=s | loss / validation 使用哪版参数 |
|---|---|---|---|
| 1 | 1 | 更新后 s+1 | 训练 loss：`θ_s`；本轮无 validation |
| 2 | 2 | 更新后 s+2 | 训练 loss：`θ_(s+1)`；validation：`θ_(s+2)` |

`record["update"]` 来自本次 `range(1, updates+1)`，再次调用 `run_training` 会从 1 重新编号；**Adam 的 `optimizer.state["step"]` 则在原对象中继续累计**，两者不可混为同一个进度字段。`train_rng` 与 `val_rng` 是两个独立的 `random.Random` 对象，**不是 MLX random key**；验证采样不会消费训练 RNG。

实际源码先调用 `train_step` 完成 Adam 更新，再生成当前 `record`，然后视 `val_every` 执行验证；验证成功后才 `records.append(record)`。若验证途中报错，`estimate_validation_loss` 的 `finally` 恢复训练模式，但**此前 Adam 更新不会自动回滚**；当次 record 尚未 append，整个函数可能直接抛异常而不返回 history。此示例没有运行，不声称上述数值是观测结果。

**2026-11-18 独立复核首次教学修订：1/2；未执行。**

### 23.3 调用者负责的对象身份契约（Caller-side identity contract）

`make_train_step(model, optimizer)` 在内部调用 `nn.value_and_grad(model, language_model_loss)`，返回的 `step` **闭包（closure）捕获这一 `model` 对象和对应 `optimizer`**；`run_training(model, train_step, ...)` 则用传入的 `model` 切换训练模式、执行验证。因此调用者必须保证两处传入的是**同一个模型实例**（`is` 的对象身份，而不仅是相同类别或 shape），且 optimizer 属于这一模型。`run_training` **不会检查** `train_step` 闭包绑定的是哪个模型；误把针对模型 A 创建的 step 与模型 B 一起传入，可能更新 A 却验证 B。

`train_rng` 与 `val_rng` 必须由调用者分别创建为**两个不同的 `random.Random` 对象**（例如前文 `Random(2026)`、`Random(2027)`，要求 `train_rng is not val_rng`）。`run_training` 仅在提供 `val_tokens` 且 `val_rng is None` 时做缺失检查，**不核验两套 RNG 的对象身份，也不主动验证闭包绑定**；下游 `get_batch` 会检查传入 RNG 的类型，但同一个对象传给训练与验证仍能通过类型检查，并让两个阶段共用、推进同一随机状态。这里明确的是**调用者前置条件**，并没有为 `run_training` 添加新的防御性检查或运行任何训练。

**独立复核第二次教学修订：23.3 累计 2/2（保留原首次 1/2 记录）；本次仅 Markdown，未执行。**
